# 2단계 실험: 수집 자료 → RAG 저장 (Chroma)

1단계에서 수집한 `ResearchResult` 의 source content 를 chunking → OpenAI 임베딩 → Chroma 에 영속 저장합니다.

**준비**: `.env` 에 `OPENAI_API_KEY`, `TAVILY_API_KEY`. `data/chroma/` 디렉토리는 자동 생성됩니다.

## 0. 환경 설정 (CWD 를 프로젝트 루트로)

Chroma persist_dir 가 `data/chroma` 상대 경로이므로, 이후 셀들은 프로젝트 루트에서 실행되는 것을 가정합니다.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    print(f"{k}: {'OK' if os.getenv(k) else 'MISSING'}")
print("CWD:", Path.cwd())

## 1. 1단계 결과 받아오기

1단계 노트북을 따로 돌릴 필요 없이 여기서 새 주제로 검색합니다.

In [ ]:
from llm_jacky.research import run_research

TOPIC = "외국인을 위한 한국 길거리 음식 추천"
result = run_research(TOPIC, max_results=5)
print(f"sources: {len(result.sources)}")
for s in result.sources:
    print(" -", s.title[:60], "|", len(s.content), "chars")

## 2. Chunking 미리 보기

Chroma 에 저장하기 전에 어떻게 잘리는지 확인합니다.

In [ ]:
from llm_jacky.rag import _to_documents

docs = _to_documents(result)
print(f"chunks: {len(docs)}\n")
for d in docs[:3]:
    print("meta:", {k: v for k, v in d.metadata.items() if k != 'topic'})
    print("text:", d.page_content[:160].replace('\n', ' '), "...\n")

## 3. 인덱싱 실행 (Chroma 에 영속 저장)

동일 `(url, chunk_index)` 는 dedupe 됩니다. 다시 실행해도 중복 누적되지 않습니다.

In [ ]:
from llm_jacky.rag import index_research

n = index_research(result)
print(f"indexed chunks: {n}")
print("persist dir:", (Path.cwd() / "data" / "chroma").resolve())

## 4. 유사도 검색 (raw)

쿼리에 대해 어떤 chunk 가 검색되는지 확인합니다.

In [ ]:
from llm_jacky.rag import _vector_store

vs = _vector_store()
QUERY = "매운 떡볶이를 처음 먹는 외국인에게 어떻게 설명할까?"

hits = vs.similarity_search_with_score(QUERY, k=4)
for doc, score in hits:
    print(f"score={score:.3f} | {doc.metadata.get('title','')[:60]}")
    print(f"  url: {doc.metadata.get('url')}")
    print(f"  text: {doc.page_content[:200].strip()}\n")

## 5. Retriever 인터페이스

3단계(Writing Chain) 에서 LCEL 로 연결할 때 그대로 사용할 형태입니다.

In [ ]:
from llm_jacky.rag import get_retriever

retriever = get_retriever(k=3)
retrieved = retriever.invoke(QUERY)
for d in retrieved:
    print("-", d.metadata.get("title", "")[:60], "|", d.metadata.get("url"))

## 6. 컬렉션 상태 / 정리

필요시 컬렉션을 비울 수 있습니다 (실험 리셋용).

In [ ]:
vs = _vector_store()
data = vs.get()
print("total docs in collection:", len(data.get("ids", [])))

# 리셋이 필요하면 아래 한 줄 주석 해제:
# vs.delete_collection()